# YOLO11m Cross-Domain Training

Train YOLO11m on two cross-domain scenarios:

1. **Scenario 1 – Degradation shift:** Train on all except DeepFish, test on DeepFish
2. **Scenario 2 – Colour/habitat shift:** Train on all except luderick, test on luderick

Scenarios should be prepared under `cross_domain_scenarios_datasets/`.

## 1. Environment Setup

In [1]:
# If needed, install ultralytics (uncomment the following line):
# %pip install ultralytics

import os
from pathlib import Path

from ultralytics import YOLO

# Project root (defaults to the current working directory).
# Optionally set HT_VISION_ROOT to point to your project folder.
ROOT = Path(os.environ.get("HT_VISION_ROOT", Path.cwd())).resolve()
# Scenarios directory (optionally override with HT_VISION_SCENARIOS_ROOT)
SCENARIOS_ROOT = Path(os.environ.get("HT_VISION_SCENARIOS_ROOT", ROOT / "cross_domain_scenarios_datasets")).resolve()

# Scenario directories
SCENARIO1_NAME = "scenario1_all_except_deepfish_test_deepfish"
SCENARIO2_NAME = "scenario2_all_except_luderick_test_luderick"

SCENARIO1_ROOT = SCENARIOS_ROOT / SCENARIO1_NAME
SCENARIO2_ROOT = SCENARIOS_ROOT / SCENARIO2_NAME

SCENARIO1_DATA = SCENARIO1_ROOT / "data.yaml"
SCENARIO2_DATA = SCENARIO2_ROOT / "data.yaml"

print("Scenario 1 data.yaml:", SCENARIO1_DATA)
print("Scenario 2 data.yaml:", SCENARIO2_DATA)

Scenario 1 data.yaml: /mnt/Data1/mpiccolo/HT_Vision/Controlled_cross_domain_generalization/cross_domain_scenarios_datasets/scenario1_all_except_deepfish_test_deepfish/data.yaml
Scenario 2 data.yaml: /mnt/Data1/mpiccolo/HT_Vision/Controlled_cross_domain_generalization/cross_domain_scenarios_datasets/scenario2_all_except_luderick_test_luderick/data.yaml


## 2. Training Configuration

Set global hyperparameters and output paths.

In [ ]:
# Model and output settings
MODEL_NAME = "yolo11m.pt"
PROJECT_ROOT = ROOT / "yolo11m_cross_domain_runs"
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

# Training hyperparameters
EPOCHS = 100
BATCH = 16
IMG_SIZE = 640
WORKERS = 8
DEVICE = 0
SEED = 42

print("Project runs root:", PROJECT_ROOT)

# Optimized hyperparameters
BEST_OPTIMIZER = "SGD"
BEST_DROPOUT = 0.1
BEST_LR0 = 0.00164
BEST_BOX_WEIGHT = 5.0
BEST_CLS_WEIGHT = 0.4
BEST_OBJ_WEIGHT = 1.0

# Augmentation settings
BEST_MOSAIC = 0.9129
BEST_MIXUP = 0.4553
BEST_FLIPUD = 0.07835
BEST_FLIPLR = 0.5
BEST_HSV_H = 0.0083
BEST_HSV_S = 0.02738
BEST_HSV_V = 0.33474

Project runs root: /mnt/Data1/mpiccolo/HT_Vision/Controlled_cross_domain_generalization/yolo11m_cross_domain_runs


## 3. Scenario Sanity Check

Count images in each split for both scenarios.

In [3]:
def count_images_in_split(scen_root: Path, split: str) -> int:
    img_dir = scen_root / "images" / split
    exts = {".jpg", ".jpeg", ".png"}
    return sum(1 for p in img_dir.glob("*") if p.suffix.lower() in exts)

for name, root in [(SCENARIO1_NAME, SCENARIO1_ROOT), (SCENARIO2_NAME, SCENARIO2_ROOT)]:
    print(f"\n{name}:")
    for split in ["train", "val", "test"]:
        n = count_images_in_split(root, split)
        print(f"  {split}: {n} images")


scenario1_all_except_deepfish_test_deepfish:
  train: 17797 images
  val: 4451 images
  test: 6517 images

scenario2_all_except_luderick_test_luderick:
  train: 19590 images
  val: 4899 images
  test: 4276 images


## 4. Train Scenario 1 (all except DeepFish → test on DeepFish)

In [4]:
# ------------------------------------
# Train YOLO11m on Scenario 1
# ------------------------------------
model_scen1 = YOLO(MODEL_NAME)

results_scen1 = model_scen1.train(
    data=str(SCENARIO1_DATA),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMG_SIZE,
    project=str(PROJECT_ROOT),
    name="scenario1_yolo11m",
    device=DEVICE,
    workers=WORKERS,
    seed=SEED,
    verbose=True,

    # ---------------------------
    # Best optimizer & loss params
    # ---------------------------
    optimizer=BEST_OPTIMIZER,
    lr0=BEST_LR0,
    dropout=BEST_DROPOUT,
    box=BEST_BOX_WEIGHT,
    cls=BEST_CLS_WEIGHT,
    kobj=BEST_OBJ_WEIGHT,

    # ---------------------------
    # Best augmentation settings
    # ---------------------------
    mosaic=BEST_MOSAIC,
    mixup=BEST_MIXUP,
    flipud=BEST_FLIPUD,
    fliplr=BEST_FLIPLR,
    hsv_h=BEST_HSV_H,
    hsv_s=BEST_HSV_S,
    hsv_v=BEST_HSV_V,
)

print("Scenario 1 training completed. Last run directory:", results_scen1.save_dir)


New https://pypi.org/project/ultralytics/8.3.235 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.197 🚀 Python-3.9.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 15840MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=5.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.4, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/mnt/Data1/mpiccolo/HT_Vision/Controlled_cross_domain_generalization/cross_domain_scenarios_datasets/scenario1_all_except_deepfish_test_deepfish/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.07835, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0083, hsv_s=0.02738, hsv_v=0.33474, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00164, lrf=0.01,

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



     75/100      7.96G      0.816      0.639      1.193         35        640: 100% ━━━━━━━━━━━━ 1113/1113 6.4it/s 2:53<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 140/140 7.9it/s 17.7s0.1s
                   all       4451      12086       0.89      0.826      0.896        0.6

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     76/100      8.06G     0.8139     0.6405      1.189         47        640: 100% ━━━━━━━━━━━━ 1113/1113 6.4it/s 2:53<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 140/140 7.9it/s 17.7s0.1s
                   all       4451      12086       0.89      0.826      0.896      0.601

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     77/100      8.04G     0.8065     0.6291      1.184        108        640: 99% ━━━━━━━━━━━╸ 1105/1113 6.3it/s 2:52<1.3ss

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



## 5. Validate Scenario 1 Model

In [13]:
# ------------------------------------
# Validate Scenario 1 Best Weights
# ------------------------------------

# Load best weights from Scenario 1
best_scen1 = YOLO(str(PROJECT_ROOT / "scenario1_yolo11m" / "weights" / "best.pt"))

val_results_scen1 = best_scen1.val(
    data=str(SCENARIO1_DATA),
    split="val",
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    verbose=True,
)

for k, v in val_results_scen1.results_dict.items():
    print(f"{k}: {v:.4f}")


Ultralytics 8.3.197 🚀 Python-3.9.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 15840MiB)
YOLO11m summary (fused): 125 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1956.2±782.8 MB/s, size: 21.8 KB)
val: Scanning /mnt/Data1/mpiccolo/HT_Vision/Controlled_cross_domain_generalization/cross_domain_scenarios_datasets/scenario1_all_except_deepfish_test_deepfish/labels/val.cache... 4451 images, 52 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4451/4451 9.0Mit/s 0.0s0s
val: /mnt/Data1/mpiccolo/HT_Vision/Controlled_cross_domain_generalization/cross_domain_scenarios_datasets/scenario1_all_except_deepfish_test_deepfish/images/val/fishclef_06542.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 279/279 11.6it/s 24.0s<0.1s
                   all       4451      12086      0.891      0.835      0.898      0.604
Speed: 0.1ms preprocess, 3.6ms infe

## 6. Test Scenario 1 (DeepFish domain)

In [14]:
# ------------------------------------
# Test Scenario 1 Best Weights
# ------------------------------------

test_results_scen1 = best_scen1.val(
    data=str(SCENARIO1_DATA),
    split="test",
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    verbose=True,
)

for k, v in test_results_scen1.results_dict.items():
    print(f"{k}: {v:.4f}")


Ultralytics 8.3.197 🚀 Python-3.9.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 15840MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 562.1±1193.0 MB/s, size: 183.5 KB)
val: Scanning /mnt/Data1/mpiccolo/HT_Vision/Controlled_cross_domain_generalization/cross_domain_scenarios_datasets/scenario1_all_except_deepfish_test_deepfish/labels/test.cache... 6517 images, 2012 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 6517/6517 16.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 408/408 14.5it/s 28.2s<0.1s
                   all       6517      15463      0.616      0.552      0.511      0.285
Speed: 0.1ms preprocess, 2.4ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /mnt/Data1/mpiccolo/HT_Vision/Controlled_cross_domain_generalization/runs/detect/val7
metrics/precision(B): 0.6157
metrics/recall(B): 0.5524
metrics/mAP50(B): 0.5112
metrics/mAP50-95(B): 0.2852
fitness: 0.3078


## 7. Train Scenario 2 (all except luderick → test on luderick)

In [ ]:
# ------------------------------------
# Train YOLO11m on Scenario 2
# ------------------------------------
model_scen2 = YOLO(MODEL_NAME)

results_scen2 = model_scen2.train(
    data=str(SCENARIO2_DATA),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMG_SIZE,
    project=str(PROJECT_ROOT),
    name="scenario2_yolo11m",
    device=DEVICE,
    workers=WORKERS,
    seed=SEED,
    verbose=True,

    # ---------------------------
    # Best optimizer & loss params
    # ---------------------------
    optimizer=BEST_OPTIMIZER,
    lr0=BEST_LR0,
    dropout=BEST_DROPOUT,
    box=BEST_BOX_WEIGHT,
    cls=BEST_CLS_WEIGHT,
    kobj=BEST_OBJ_WEIGHT,

    # ---------------------------
    # Best augmentation settings
    # ---------------------------
    mosaic=BEST_MOSAIC,
    mixup=BEST_MIXUP,
    flipud=BEST_FLIPUD,
    fliplr=BEST_FLIPLR,
    hsv_h=BEST_HSV_H,
    hsv_s=BEST_HSV_S,
    hsv_v=BEST_HSV_V,
)

print("Scenario 2 training completed. Last run directory:", results_scen2.save_dir)


New https://pypi.org/project/ultralytics/8.3.235 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.197 🚀 Python-3.9.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 15840MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=5.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.4, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/mnt/Data1/mpiccolo/HT_Vision/Controlled_cross_domain_generalization/cross_domain_scenarios_datasets/scenario2_all_except_luderick_test_luderick/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.07835, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0083, hsv_s=0.02738, hsv_v=0.33474, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00164, lrf=0.01,

## 8. Validate Scenario 2 model on validation set

In [9]:
# ------------------------------------
# Validate Scenario 2 Best Weights
# ------------------------------------

best_scen2 = YOLO(str(PROJECT_ROOT / "scenario2_yolo11m" / "weights" / "best.pt"))

val_results_scen2 = best_scen2.val(
    data=str(SCENARIO2_DATA),
    split="val",
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    verbose=True,
)

for k, v in val_results_scen2.results_dict.items():
    print(f"{k}: {v:.4f}")


Ultralytics 8.3.197 🚀 Python-3.9.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 15840MiB)
YOLO11m summary (fused): 125 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2489.3±2268.9 MB/s, size: 52.9 KB)
val: Scanning /mnt/Data1/mpiccolo/HT_Vision/Controlled_cross_domain_generalization/cross_domain_scenarios_datasets/scenario2_all_except_luderick_test_luderick/labels/val.cache... 4899 images, 454 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4899/4899 11.7Mit/s 0.0ss
val: /mnt/Data1/mpiccolo/HT_Vision/Controlled_cross_domain_generalization/cross_domain_scenarios_datasets/scenario2_all_except_luderick_test_luderick/images/val/fishclef_06542.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 307/307 13.1it/s 23.5s0.2ss
                   all       4899      13289      0.899       0.85      0.909      0.608
Speed: 0.1ms preprocess, 3.1ms in

## 9. Evaluate Scenario 2 model on test set (luderick domain)

Run evaluation on the test split to measure **colour/habitat-shift performance** on luderick.

In [10]:
# ------------------------------------
# Test Scenario 2 Best Weights
# ------------------------------------

test_results_scen2 = best_scen2.val(
    data=str(SCENARIO2_DATA),
    split="test",
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    verbose=True,
)

for k, v in test_results_scen2.results_dict.items():
    print(f"{k}: {v:.4f}")


Ultralytics 8.3.197 🚀 Python-3.9.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 15840MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 372.8±822.3 MB/s, size: 45.8 KB)
val: Scanning /mnt/Data1/mpiccolo/HT_Vision/Controlled_cross_domain_generalization/cross_domain_scenarios_datasets/scenario2_all_except_luderick_test_luderick/labels/test... 4276 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4276/4276 1.4Kit/s 3.0s<0.0s
val: New cache created: /mnt/Data1/mpiccolo/HT_Vision/Controlled_cross_domain_generalization/cross_domain_scenarios_datasets/scenario2_all_except_luderick_test_luderick/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 268/268 10.0it/s 26.8s0.1ss
                   all       4276       9429      0.752      0.698      0.754      0.497
Speed: 0.1ms preprocess, 4.3ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to /mnt/Data1/mpiccolo/HT_Vision/Controlled_cross

## 10. Comparison Table between two scenrios

In [12]:
import pandas as pd

# Create comparison table between Scenario 1 and Scenario 2
comparison_data = {
    'Metric': ['mAP50 (Val)', 'mAP50-95 (Val)', 'Precision (Val)', 'Recall (Val)',
               'mAP50 (Test)', 'mAP50-95 (Test)', 'Precision (Test)', 'Recall (Test)'],
    'Scenario 1 (excl. DeepFish)': [
        val_results_scen1.box.map50,
        val_results_scen1.box.map,
        val_results_scen1.box.mp,
        val_results_scen1.box.mr,
        test_results_scen1.box.map50,
        test_results_scen1.box.map,
        test_results_scen1.box.mp,
        test_results_scen1.box.mr
    ],
    'Scenario 2 (excl. luderick)': [
        val_results_scen2.box.map50,
        val_results_scen2.box.map,
        val_results_scen2.box.mp,
        val_results_scen2.box.mr,
        test_results_scen2.box.map50,
        test_results_scen2.box.map,
        test_results_scen2.box.mp,
        test_results_scen2.box.mr
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("=" * 80)
print("CROSS-DOMAIN TRAINING RESULTS COMPARISON")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)
print(f"\nScenario 1: Trained on all domains except DeepFish, tested on DeepFish (degradation shift)")
print(f"Scenario 2: Trained on all domains except luderick, tested on luderick (colour/habitat shift)")
print("=" * 80)

CROSS-DOMAIN TRAINING RESULTS COMPARISON
          Metric  Scenario 1 (excl. DeepFish)  Scenario 2 (excl. luderick)
     mAP50 (Val)                     0.897825                     0.908966
  mAP50-95 (Val)                     0.604385                     0.607895
 Precision (Val)                     0.891182                     0.899041
    Recall (Val)                     0.835098                     0.850403
    mAP50 (Test)                     0.511218                     0.753516
 mAP50-95 (Test)                     0.285218                     0.497381
Precision (Test)                     0.615739                     0.752068
   Recall (Test)                     0.552439                     0.698103

Scenario 1: Trained on all domains except DeepFish, tested on DeepFish (degradation shift)
Scenario 2: Trained on all domains except luderick, tested on luderick (colour/habitat shift)


## 11. Export per-image test results to CSV

Generate detailed per-image metrics for both test scenarios and save them as CSV files for downstream analysis.

In [17]:
import json
import numpy as np
import torch

def compute_iou(box1, box2):
    """Compute IoU between two boxes in xyxy format."""
    x1_min, y1_min, x1_max, y1_max = box1
    x2_min, y2_min, x2_max, y2_max = box2
    
    inter_xmin = max(x1_min, x2_min)
    inter_ymin = max(y1_min, y2_min)
    inter_xmax = min(x1_max, x2_max)
    inter_ymax = min(y1_max, y2_max)
    
    inter_area = max(0, inter_xmax - inter_xmin) * max(0, inter_ymax - inter_ymin)
    
    box1_area = (x1_max - x1_min) * (y1_max - y1_min)
    box2_area = (x2_max - x2_min) * (y2_max - y2_min)
    
    union_area = box1_area + box2_area - inter_area
    
    if union_area == 0:
        return 0.0
    return inter_area / union_area


def parse_yolo_label(label_path, img_width, img_height):
    """Parse YOLO format label file and convert to xyxy format."""
    boxes = []
    if not label_path.exists():
        return boxes
    
    with open(label_path, 'r') as f:
        lines = f.read().strip().split('\n')
    
    for line in lines:
        if not line.strip():
            continue
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        
        # YOLO format: class x_center y_center width height (normalized)
        cls, x_center, y_center, width, height = map(float, parts[:5])
        
        # Convert to pixel coordinates
        x_center *= img_width
        y_center *= img_height
        width *= img_width
        height *= img_height
        
        # Convert to xyxy format
        x_min = x_center - width / 2
        y_min = y_center - height / 2
        x_max = x_center + width / 2
        y_max = y_center + height / 2
        
        boxes.append([x_min, y_min, x_max, y_max])
    
    return boxes


def match_predictions_to_gt(pred_boxes, gt_boxes, iou_threshold=0.5):
    """Match predictions to ground truth boxes using IoU threshold.
    Returns: tp, fp, fn, matched_ious, matched_confs
    """
    if len(gt_boxes) == 0:
        # No GT: all predictions are FP
        fp = len(pred_boxes)
        return 0, fp, 0, [], [conf for _, conf in pred_boxes]
    
    if len(pred_boxes) == 0:
        # No predictions: all GT are FN
        fn = len(gt_boxes)
        return 0, 0, fn, [], []
    
    # Compute IoU matrix
    iou_matrix = np.zeros((len(pred_boxes), len(gt_boxes)))
    for i, (pred_box, _) in enumerate(pred_boxes):
        for j, gt_box in enumerate(gt_boxes):
            iou_matrix[i, j] = compute_iou(pred_box, gt_box)
    
    # Greedy matching: match predictions to GT in order of confidence
    matched_gt = set()
    tp = 0
    fp = 0
    matched_ious = []
    matched_confs = []
    
    for i, (pred_box, conf) in enumerate(pred_boxes):
        # Find best matching GT
        best_iou = 0
        best_gt_idx = -1
        for j in range(len(gt_boxes)):
            if j not in matched_gt and iou_matrix[i, j] > best_iou:
                best_iou = iou_matrix[i, j]
                best_gt_idx = j
        
        if best_iou >= iou_threshold:
            tp += 1
            matched_gt.add(best_gt_idx)
            matched_ious.append(best_iou)
            matched_confs.append(conf)
        else:
            fp += 1
    
    fn = len(gt_boxes) - len(matched_gt)
    
    return tp, fp, fn, matched_ious, matched_confs


def export_per_image_test_results(
    model,
    scenario_root: Path,
    scenario_name: str,
    model_name: str,
    imgsz: int,
    batch: int,
    device: int,
    output_csv: Path
):
    """
    Run predictions on test images and export per-image metrics to CSV.
    
    For each test image, computes:
    - Precision, Recall, F1 per image
    - TP, FP, FN counts
    - Mean confidence and IoU
    """
    
    # Get test images directory
    test_img_dir = scenario_root / "images" / "test"
    test_lbl_dir = scenario_root / "labels" / "test"
    
    # Get all test images
    test_images = sorted([
        p for p in test_img_dir.glob("*")
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ])
    
    print(f"Processing {len(test_images)} test images for {scenario_name}...")
    
    results_list = []
    
    for img_path in test_images:
        # Run prediction on single image
        result = model.predict(
            source=str(img_path),
            imgsz=imgsz,
            conf=0.25,  # default confidence threshold
            device=device,
            verbose=False
        )[0]
        
        # Get image dimensions
        img_height, img_width = result.orig_shape
        
        # Parse ground truth boxes
        label_path = test_lbl_dir / img_path.with_suffix(".txt").name
        gt_boxes = parse_yolo_label(label_path, img_width, img_height)
        
        # Extract predictions (xyxy format)
        pred_boxes = []
        if len(result.boxes) > 0:
            boxes_xyxy = result.boxes.xyxy.cpu().numpy()
            confidences = result.boxes.conf.cpu().numpy()
            
            for box, conf in zip(boxes_xyxy, confidences):
                pred_boxes.append((box.tolist(), float(conf)))
        
        # Match predictions to GT
        tp, fp, fn, matched_ious, matched_confs = match_predictions_to_gt(pred_boxes, gt_boxes)
        
        # Compute per-image metrics
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        
        # Compute mean IoU and confidence for matched predictions
        iou_mean = float(np.mean(matched_ious)) if matched_ious else 0.0
        confidence_mean = float(np.mean(matched_confs)) if matched_confs else 0.0
        
        results_list.append({
            'image_path': str(img_path),
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'tp': tp,
            'fp': fp,
            'fn': fn,
            'confidence_mean': confidence_mean,
            'iou_mean': iou_mean,
            'model_name': model_name,
            'scenario_name': scenario_name,
        })
    
    # Create DataFrame and save
    df = pd.DataFrame(results_list)
    df.to_csv(output_csv, index=False)
    print(f"Saved per-image results to: {output_csv}")
    print(f"Total images: {len(df)}")
    print(f"Mean Precision: {df['precision'].mean():.4f}")
    print(f"Mean Recall: {df['recall'].mean():.4f}")
    print(f"Mean F1: {df['f1'].mean():.4f}")
    print(f"Total TP: {df['tp'].sum()}, FP: {df['fp'].sum()}, FN: {df['fn'].sum()}")
    
    return df


# Export Scenario 1 test results
csv_scen1 = ROOT / "per_image_results_scenario1_test.csv"
df_scen1 = export_per_image_test_results(
    model=best_scen1,
    scenario_root=SCENARIO1_ROOT,
    scenario_name=SCENARIO1_NAME,
    model_name="yolo11m",
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    output_csv=csv_scen1
)

print("\n" + "="*80 + "\n")

# Export Scenario 2 test results
csv_scen2 = ROOT / "per_image_results_scenario2_test.csv"
df_scen2 = export_per_image_test_results(
    model=best_scen2,
    scenario_root=SCENARIO2_ROOT,
    scenario_name=SCENARIO2_NAME,
    model_name="yolo11m",
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    output_csv=csv_scen2
)

Processing 6517 test images for scenario1_all_except_deepfish_test_deepfish...
Saved per-image results to: /mnt/Data1/mpiccolo/HT_Vision/Controlled_cross_domain_generalization/per_image_results_scenario1_test.csv
Total images: 6517
Mean Precision: 0.4080
Mean Recall: 0.4417
Mean F1: 0.3984
Total TP: 9314, FP: 7946, FN: 6149


Processing 4276 test images for scenario2_all_except_luderick_test_luderick...
Saved per-image results to: /mnt/Data1/mpiccolo/HT_Vision/Controlled_cross_domain_generalization/per_image_results_scenario2_test.csv
Total images: 4276
Mean Precision: 0.5000
Mean Recall: 0.8281
Mean F1: 0.5727
Total TP: 8056, FP: 11976, FN: 1373


## 12. Summary and next steps

This notebook has:
- Trained YOLO11m on two cross-domain scenarios.
- Validated the models on in-domain validation sets.
- Evaluated the models on held-out domain-shift test sets (DeepFish and luderick).
- **Exported per-image test results to CSV files:**
  - `per_image_results_scenario1_test.csv`
  - `per_image_results_scenario2_test.csv`

The next steps for your project are:
- Merge the per-image metrics with the distortion features to run correlation analyses.
- Generate plots and tables for the thesis (e.g., Chapter 4.7).